In [1]:
import numpy as np
from sympy import symbols, Eq, solve
import csv
import pandas as pd
 
t,s = symbols('t s')
 
#Parameters?
 
def midpoint(x1,y1,x2,y2):
    num_1 = x1 + x2
    num_2 = y1 + y2
    points = np.array([num_1/2, num_2/2])
    print("points: ", points)
    return points
 
def distance(x1, y1, x2, y2):
    d = np.sqrt((x2 - x1)**2 + (y2 - y1)**2)
    print("distance: ", d)
    return d
 
#(-dy, dx). Flip and make dy negative and multiply times 1/4
def perpendicular(x2, y2):
    flip_array = np.array([-y2, x2])
    quarter_d = 0.25*flip_array
    #print("flipped array: ", flip_array)
    print("quarter distance: ", quarter_d)
    return quarter_d
 
def new_perpendicular(x2, y2):
    flip_array = np.array([-y2, x2])
    print("flipped array: ", flip_array)
    return flip_array
 
def radius(x1, y1, h, k):
    r_square = (x1 - h)**2 + (y1 - k)**2
    r = np.sqrt(r_square)
    print("r:", r)
    return r
 
def final_vel (v, r, x1, y1, x2, y2, h, k):
    if r == 0: #  Cannot have a 0 in the denominator
        return 0
   
    dx = x2 - x1
    dy = y2 - y1
 
    #Rosbot to the center of the cirlce
    cx = h - x1 #Vector
    cy = k - y1 #Vector
 
    #Need the crossproduct to determine the direction. Otherwise velocity is always positive
    cross = dx * cy - dy * cx
    if cross > 0:
        sign = 1
    else:
        sign = -1
 
    result = sign * v/r
    print("final angular velocity:", result)
    return result
 
# Load only the 'Name' and 'Age' columns
df = pd.read_csv('distance_calculator_v2.csv',)
print(df.columns.tolist())
print(df.head())
 
#Array to store rsesults and for .csv file
results = []
 
 
for idx, row in df.iterrows():
 
    #Need to convert everything to floats
    x1 = row['RobotX']
    print("x1:", x1)
 
    y1 = row['RobotY']
    print("y1", y1)
 
    x2 = row['AmbX']
    print("x2", x2)
 
    y2 = row['AmbY']
    print("y2", y2)
 
    #Calling all the functions to get the right math
    p1 = np.array([x1, y1])
    p2 = np.array([x2, y2])
 
    chord1 = p2 - p1
 
    #From Lauren's example
    s1 = midpoint(x1 ,y1, x2, y2)
    #print("should get (1.054, 0.0185)")
    #Should get (1.054, 0.0185)
 
    #Distance from p1 to p2
    d1 = distance(x1, y1, x2, y2)
    #Should get 2.107
    #print("should get 2.107")
 
    #Perpendicular test
    pe1 = perpendicular(x2, y2)
    #Should get (-0.00925, 0.52675)
    #print("should get (-0.00925, 0.52675)")
 
    p3 = s1 + pe1
    print("p3:", p3)
    #should get (1.04425, 0.54525)
    #print("should get (1.04425, 0.54525)")
 
    #For building the triangle
 
    #Midpoint of chord p1 -> p2
 
    #Midpoint of chord p1 -> p3
    s2 = midpoint(x1 ,y1, p3[0], p3[1])
    #should get (0.522, 0.273)
    #print("should get (0.522, 0.273)")
 
    n1 = new_perpendicular(chord1[0], chord1[1])
    #Need to double check
    #should get (-0.037, 2.107)
    #print("should get (-0.037, 2.107)")
 
    n2 = new_perpendicular(p3[0], p3[1])
    #should get (-0.545, 1.044)
    #print("should get (-0.545, 1.044)")
 
    ray_1 = s1 + n1*t
    print(ray_1[0])
    print(ray_1[1])
    #should get (1.054 - 0.037t, 0.018s + 2.107t)
    #print("(1.054 - 0.037t, 0.0185 + 2.107t)")
 
    ray_2 = s2 + n2*s
    print(ray_2[0])
    print(ray_2[1])
    #should get (0.522 - 0.546s, 0.273 + 1.045s)
    #print("should get (0.522 - 0.546s, 0.273 + 1.045s)")
 
    #Set x's equal
    ray1_eq = Eq(ray_1[0], ray_2[0]) #For finding the x's
 
    #Set y's equal
    ray2_eq = Eq(ray_1[1], ray_2[1]) #For finding the x's
 
    #Solve for unknowns
    #find t and s
    solution = solve((ray1_eq, ray2_eq), (t,s))
 
    if isinstance(solution, list):
        if len(solution) == 0:
            continue #If case in case there is no solution for the circle
        solution = solution[0]
    #Returns an array
 
    #Solvers
 
    #Then plug back into center:
    t_variable = solution[t]
    s_variable = solution[s]
 
    center = s2 + n2*s_variable
    print("center: ", center)
 
    #Should give out (1.068, -0.772) which belong to h and k respectively
    #print("Should give out (1.068, -0.772)")
    h, k = center
    #cast these
    h = float(h)
    k = float(k)
 
    r = radius(x1, y1, h, k)
    print("radius: ", r)
 
    v = 0.5
    result = final_vel(v, r, x1, y1, x2, y2, h, k)
 
df.to_csv('output_results.csv', index=False)

FileNotFoundError: [Errno 2] No such file or directory: 'distance_calculator_v2.csv'